In [0]:
import requests
import pandas as pd
from pyspark.sql import functions as F

LOCATIONS = {
    "York": (53.96, -1.08),
    "Leeds": (53.80, -1.55),
    "Bristol": (51.45, -2.59),
    "Exeter": (50.72, -3.53),
}

In [0]:
def extract(city, lat, lon):
    url = (
        "https://api.open-meteo.com/v1/forecast"
        f"?latitude={lat}&longitude={lon}"
        "&daily=temperature_2m_max,temperature_2m_min,precipitation_sum"
        "&past_days=14&forecast_days=1&timezone=Europe/London"
    )
    response = requests.get(url)
    response.raise_for_status()
    pdf = pd.DataFrame(response.json()["daily"])   # pandas DataFrame
    pdf["city"] = city
    return pdf

frames = [extract(city, lat, lon) for city, (lat, lon) in LOCATIONS.items()]
raw_pdf = pd.concat(frames, ignore_index=True)

raw = spark.createDataFrame(raw_pdf)    # convert pandas into Spark
raw.printSchema()
raw.show(5)

root
 |-- time: string (nullable = true)
 |-- temperature_2m_max: double (nullable = true)
 |-- temperature_2m_min: double (nullable = true)
 |-- precipitation_sum: double (nullable = true)
 |-- city: string (nullable = true)

+----------+------------------+------------------+-----------------+----+
|      time|temperature_2m_max|temperature_2m_min|precipitation_sum|city|
+----------+------------------+------------------+-----------------+----+
|2026-11-01|              11.6|               4.2|              0.0|York|
|2026-11-02|              11.8|               5.7|              0.4|York|
|2026-11-03|              11.5|               8.5|              0.1|York|
|2026-11-04|              12.6|               9.2|              1.1|York|
|2026-11-05|              11.7|              10.0|              0.4|York|
+----------+------------------+------------------+-----------------+----+
only showing top 5 rows


In [0]:
def transform(df):
    # TODO 1: rename the columns: time→date, temperature_2m_max→max_temp,
    #         temperature_2m_min→min_temp, precipitation_sum→rain_mm
    #         (hint: .withColumnRenamed("old", "new"), chained 4 times)
    df = df.withColumnRenamed("time","date").withColumnRenamed("temperature_2m_max","max_temp").withColumnRenamed("temperature_2m_min","min_temp").withColumnRenamed("precipitation_sum","rain_mm")
    # TODO 2: convert date to a proper date type with F.to_date
    df = df.withColumn("date", F.to_date(F.col("date")))
    # TODO 3: drop rows with missing values
    df = df.dropna()
    # TODO 4: add 'rain_day': True if rain_mm > 1
    df = df.withColumn("rain_day", F.col("rain_mm") > 1)
    # TODO 5: add 'workable': True if rain_mm < 5 AND min_temp > 0
    df = df.withColumn("workable", (F.col("rain_mm") < 5) & (F.col("min_temp") > 0) )
    # TODO 6 (new!): add 'conditions' using F.when:
    #         rain_mm >= 5 → "Wet", min_temp <= 0 → "Frost", otherwise → "Good"
    df = df.withColumn("conditions", F.when(F.col("rain_mm") >= 5, "Wet").when(F.col("min_temp") <= 0,"Frost").otherwise ("Good"))
    return df


clean = transform(raw)
clean.show()
clean.printSchema()

+----------+--------+--------+-------+-----+--------+--------+----------+
|      date|max_temp|min_temp|rain_mm| city|rain_day|workable|conditions|
+----------+--------+--------+-------+-----+--------+--------+----------+
|2026-11-01|    11.6|     4.2|    0.0| York|   false|    true|      Good|
|2026-11-02|    11.8|     5.7|    0.4| York|   false|    true|      Good|
|2026-11-03|    11.5|     8.5|    0.1| York|   false|    true|      Good|
|2026-11-04|    12.6|     9.2|    1.1| York|    true|    true|      Good|
|2026-11-05|    11.7|    10.0|    0.4| York|   false|    true|      Good|
|2026-11-06|    11.9|     9.3|    0.0| York|   false|    true|      Good|
|2026-11-07|     9.9|     8.2|    3.9| York|    true|    true|      Good|
|2026-11-08|     9.9|     7.8|    2.5| York|    true|    true|      Good|
|2026-11-09|    10.2|     4.7|    0.0| York|   false|    true|      Good|
|2026-11-10|     9.9|     2.4|    0.0| York|   false|    true|      Good|
|2026-11-11|     9.2|     4.5|    0.0|

In [0]:
clean.write.mode("overwrite").saveAsTable("daily_weather")
print("Pipeline complete")

Pipeline complete


In [0]:
%sql
SELECT city, COUNT(*) AS total_workable FROM daily_weather WHERE workable = True GROUP BY city

city,total_workable
York,15
Leeds,15
Bristol,13
Exeter,11


In [0]:
%sql
SELECT city, SUM(CAST(workable AS INT)) AS total_workable FROM daily_weather GROUP BY city

city,total_workable
York,15
Leeds,15
Bristol,13
Exeter,11


In [0]:
%sql
SELECT city, SUM(rain_mm) AS total_rain FROM daily_weather GROUP BY city ORDER BY total_rain DESC

city,total_rain
Exeter,32.400000000000006
Bristol,22.200000000000003
York,9.3
Leeds,5.3999999999999995


In [0]:
%sql
SELECT city, date, min_temp AS lowest_temp FROM daily_weather ORDER BY lowest_temp ASC LIMIT 1

city,date,lowest_temp
Exeter,2026-11-13,-0.0


In [0]:
%sql
SELECT city, SUM(CAST(workable AS INT)) * 100.0 / COUNT(*)AS percent_workable FROM daily_weather GROUP BY city 

city,percent_workable
York,100.00000000000000
Leeds,100.00000000000000
Bristol,86.66666666666667
Exeter,73.33333333333333


In [0]:
%sql
SELECT city, conditions, COUNT(*) AS days FROM daily_weather GROUP BY city, conditions ORDER BY city

city,conditions,days
Bristol,Good,13
Bristol,Wet,2
Exeter,Frost,2
Exeter,Good,11
Exeter,Wet,2
Leeds,Good,15
York,Good,15


In [0]:
%sql
SELECT city, COUNT_IF(conditions = "Wet") AS wet_days, COUNT_IF (conditions = "Frost") AS frosty_days, COUNT_IF (conditions = "Good") AS good_days FROM daily_weather GROUP BY city

city,wet_days,frosty_days,good_days
York,0,0,15
Leeds,0,0,15
Bristol,2,0,13
Exeter,2,2,11
